In [1]:
import os
from sodapy import Socrata
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

from socrata_interface.domain import Domain

In [ ]:
SCROLL_FILE = "last_scroll_id.txt"
DOMAIN_FILE = "socrata_domains.txt"

def load_state():
    """Load scroll ID + seen domains if resuming."""
    # Load scroll ID
    if os.path.exists(SCROLL_FILE):
        with open(SCROLL_FILE, "r") as f:
            scroll_id = f.read().strip()
            if scroll_id == "":
                scroll_id = "*"   # fallback
    else:
        scroll_id = "*"

    # Load seen domains set
    seen = set()
    if os.path.exists(DOMAIN_FILE):
        with open(DOMAIN_FILE, "r") as f:
            for line in f:
                seen.add(line.strip())

    return scroll_id, seen


def save_scroll_id(scroll_id):
    """Write latest scroll ID to disk so we can resume."""
    with open(SCROLL_FILE, "w") as f:
        f.write(scroll_id)


def get_all_domains_resume():
    url = "https://api.us.socrata.com/api/catalog" # Seems to be a catalog of all things accessible by the api
    limit = 1000 # Limit 1000 because it is small enough to avoid timeouts. 10000 gets timed out. Optimal would probably be between

    # Load previous state
    scroll_id, seen = load_state()

    print(f"Starting with scroll_id={scroll_id}, {len(seen)} domains already saved.")

    # Open output file in append mode
    with open(DOMAIN_FILE, "a") as f_out:

        while True:
            print(f"Fetching scroll_id={scroll_id} ...")

            params = {"scroll_id": scroll_id, "limit": limit}

            try:
                resp = requests.get(url, params=params, timeout=10)
                resp.raise_for_status()
            except Exception as e:
                print(f"Error: {e}, retrying in 5 seconds...")
                time.sleep(5)
                continue

            data = resp.json()
            results = data.get("results", [])

            if not results:
                print("Deep scroll completed or no more results.")
                break

            # Process results
            for item in results:
                metadata = item.get("metadata", {})
                domain = metadata.get("domain")

                if domain and domain not in seen:
                    f_out.write(domain + "\n")
                    f_out.flush()  # ensure immediate write
                    seen.add(domain)

            # Update scroll ID for next request
            next_scroll = results[-1].get("resource").get("id") # id of previous resource can be used to get next scroll
            if not next_scroll:
                print("Finished scrolling dataset.")
                break

            scroll_id = next_scroll
            save_scroll_id(scroll_id)  # persist checkpoint

            time.sleep(0.2) # Only to avoid timeouts, may not be necessary

    print(f"\nCompleted with {len(seen)} total domains.")
    return seen

# Getting all domains to find cities, as the original study was about the state of urban data across US cities, not just NY
domains = get_all_domains_resume() # 555 domains discovered amongst ~220k things in the catalog, but many are for same entity

Starting with scroll_id=*, 118 domains already saved.
Fetching scroll_id=* ...
Fetching scroll_id=26is-s4fm ...
Fetching scroll_id=2auq-ndkr ...
Fetching scroll_id=2fh6-vrts ...
Fetching scroll_id=2k8a-dz2p ...
Fetching scroll_id=2qsi-qheg ...
Fetching scroll_id=2vhj-s442 ...
Fetching scroll_id=322a-riji ...
Fetching scroll_id=36ib-rtmu ...
Fetching scroll_id=3b78-mfyi ...
Fetching scroll_id=3fxc-nque ...
Fetching scroll_id=3kbj-ypat ...
Fetching scroll_id=3qys-jk4f ...
Fetching scroll_id=3vft-99rh ...
Fetching scroll_id=4292-pktu ...
Fetching scroll_id=46zs-4ngp ...
Fetching scroll_id=4bn5-jdm8 ...
Fetching scroll_id=4g6s-ak9g ...
Fetching scroll_id=4m2v-hzec ...
Fetching scroll_id=4rka-uupg ...
Fetching scroll_id=4w6e-7nqk ...
Fetching scroll_id=52ny-36z2 ...
Fetching scroll_id=576d-v5m3 ...
Fetching scroll_id=5bn2-vnxz ...
Fetching scroll_id=5g3x-yfbg ...
Fetching scroll_id=5kp7-t9c8 ...
Fetching scroll_id=5rei-mff9 ...
Fetching scroll_id=5w2u-reag ...
Fetching scroll_id=62vi-89fw .

In [7]:
def filter_lines(input_path: str, output_path: str, include: str | None = None, exclude: str | None = None):
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        
        for line in infile:
            # If include phrase is given, skip lines that don't contain it
            if include is not None and include not in line:
                continue
            
            # If exclude phrase is given, skip lines that DO contain it
            if exclude is not None and exclude in line:
                continue
            
            outfile.write(line)


In [11]:
def filter_lines_start(input_path: str, output_path: str, include_start: str | None = None, exclude_start: str | None = None):
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        
        for line in infile:
            # Check include-start condition
            if include_start is not None and not line.startswith(include_start):
                continue

            # Check exclude-start condition
            if exclude_start is not None and line.startswith(exclude_start):
                continue

            outfile.write(line)

In [16]:
filter_lines_start("socrata_domains.txt", "socrata_domains_data.txt", include_start="data.")

In [17]:
filter_lines("socrata_domains_data.txt", "socrata_domains_countyless.txt", exclude="county")

In [ ]:
def normalize_url(url: str) -> str:
    if not url.startswith(("http://", "https://")):
        return "https://" + url
    return url

def find_city(input_path: str, output_path: str, phrase: str): # Didn't even get all the cities
    print("Not city:")
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        
        for line in infile:
            text = ""
            response = requests.get(normalize_url(line.strip()))
            
            if response.ok:

                soup = BeautifulSoup(response.text, "html.parser")

                # Try meta name="title"
                meta_title = soup.find("meta", attrs={"name": "title"}) # Probably need more checks to properly find cities
                if meta_title is not None and "content" in meta_title.attrs:
                    text = meta_title["content"]

                # Fallback to the <title> tag
                elif soup.title:
                    text = soup.title.text.strip()

            if phrase in text.lower():
                outfile.write(line)
            else:
                print(line)

In [32]:
find_city("socrata_domains_countyless.txt", "socrata_domains_cities_only.txt", phrase="city")

Not city:
data.cityofnewyork.us

data.auburnwa.gov

data.cambridgema.gov

data.novascotia.ca

data.honolulu.gov

data.bayareametro.gov

data.cdc.gov

data.calgary.ca

data.bts.gov

data.pa.gov

data.edmonton.ca

data.dumfriesva.gov

data.cityofgainesville.org

data.delaware.gov

data.oce.pr.gov

data.kcmo.org

data.cincinnati-oh.gov

data.buffalony.gov

data.mmcp.ms.gov

data.wcad.org

data.memphistn.gov

data.readingpa.gov

data.framinghamma.gov

data.smcgov.org

data.vermont.gov

data.datacenterresearch.org

data.tompsc.com

data.orcities.org

data.stocktonca.gov

data.cstx.gov

data.coloradosprings.gov

data.sfgov.org

data.nhitc.org

data.fortworthtexas.gov

data.qac.org

data.oxnard.org



# Metadata needed

- Schema: "columns" | list of columns 

- Column Types: "dataTypeName" | in the list of columns

- Column Names: "name" or "fieldName" | in the list of columns

- Zipcode: "the_geom"

- Nulls: "non_null" and "null" | https://<domain>/resource/<dataset_id>.json?$select=count(*)&$where=<column_name>%20IS%20NULL

- Category: "category" | for top categories of each city

- Format: "displayType" or "viewType"

- Number of Rows: https://<domain>/resource/<dataset_id>.json?$select=count(*)

- Tags: "tags" | list of tags

- Number of Downloads: "downloadCount"

- Number of Views: "viewCount"

- Age of Dataset: "createdAt"

- Update Frequency: "indexUpdatedAt" or "rowsUpdatedAt"

In [2]:
nyc = Socrata("data.cityofnewyork.us", None, timeout=60)

In [17]:
len(nyc.datasets())

2997

In [4]:
nyc.get_metadata('erm2-nwe9')

{'id': 'erm2-nwe9',
 'name': '311 Service Requests from 2020 to Present',
 'assetType': 'dataset',
 'attribution': '311',
 'averageRating': 0,
 'category': 'Social Services',
 'createdAt': 1318225937,
 'description': '<b>NOTE:</b> Learn more about the latest changes to this dataset: https://opendata.cityofnewyork.us/311-service-requests-from-2010-to-present-updates/\n\n311 responds to thousands of inquiries, comments and requests from customers every single day. This dataset represents only service requests that can be directed to specific agencies.\n\nThis dataset is updated daily and expected values for many fields will change over time. The lists of expected values associated with each column are not exhaustive.\n\nEach row of data contains information about the service request, including complaint type, responding agency, and geographic location. However the data does not reveal any personally identifying information about the customer who made the request.\n\nFor data from 2010-20

In [4]:
nola = Socrata('data.nola.gov', None)

In [18]:
metadata = nola.get_metadata("2jgv-pqrq")

cols = metadata.get("columns") or []
relevant_columns = [
    {
        "name": c.get("fieldName"), 
        "type": c.get("dataTypeName"),
    }
    for c in cols if c.get("fieldName")
]

def _quote_field_name(field):
        """
        Quote field names that need it (contain special characters).
        """
        special_chars = {':', '@', '-', ' ', '.', '/', '\\', '(', ')'}
        if any(c in field for c in special_chars):
            escaped = field.replace('`', '``')
            return f"`{escaped}`"
        return field

select_parts = []
for col in cols:
    field = col["fieldName"]
    dtype = col["dataTypeName"]
    
    quoted_field = _quote_field_name(field)
    safe_alias = field.replace(":", "_").replace("@", "_").replace("-", "_")
    
    # null count
    select_parts.append(
        f"(count(*) - count({quoted_field})) AS {safe_alias}_nulls"
    )
    
    # Simplified semantic nulls for text fields only
    TEXT_LIKE_TYPES = {"text", "url", "email", "phone", "html"}
    if dtype in TEXT_LIKE_TYPES:
        # Simpler check - just trim and empty string
        semantic = (
            f"sum(CASE WHEN "
            f"{quoted_field} IS NULL OR "
            f"trim({quoted_field}) = '' "
            f"THEN 1 ELSE 0 END) AS {safe_alias}_semantic_nulls"
        )
    else:
        semantic = f"0 AS {safe_alias}_semantic_nulls"
    select_parts.append(semantic)

In [10]:
metadata['columns']

[{'id': 541844684,
  'name': 'Service Request #',
  'dataTypeName': 'text',
  'description': '',
  'fieldName': 'service_request',
  'position': 1,
  'renderTypeName': 'text',
  'tableColumnId': 75233504,
  'width': 100,
  'cachedContents': {'non_null': '967468',
   'largest': '2026-1257280',
   'null': '0',
   'top': [{'item': '101000002301', 'count': '1'},
    {'item': '101000002302', 'count': '1'},
    {'item': '101000002304', 'count': '1'},
    {'item': '101000002313', 'count': '1'},
    {'item': '101000002314', 'count': '1'},
    {'item': '101000002316', 'count': '1'},
    {'item': '101000002320', 'count': '1'},
    {'item': '101000002322', 'count': '1'},
    {'item': '101000002323', 'count': '1'},
    {'item': '101000002324', 'count': '1'},
    {'item': '101000002329', 'count': '1'},
    {'item': '101000002332', 'count': '1'},
    {'item': '101000002341', 'count': '1'},
    {'item': '101000002343', 'count': '1'},
    {'item': '101000002344', 'count': '1'},
    {'item': '101000002

In [4]:
select_parts

['(count(*) - count(service_request)) AS service_request_nulls',
 "sum(CASE WHEN service_request IS NULL OR trim(service_request) = '' THEN 1 ELSE 0 END) AS service_request_semantic_nulls",
 '(count(*) - count(request_type)) AS request_type_nulls',
 "sum(CASE WHEN request_type IS NULL OR trim(request_type) = '' THEN 1 ELSE 0 END) AS request_type_semantic_nulls",
 '(count(*) - count(request_reason)) AS request_reason_nulls',
 "sum(CASE WHEN request_reason IS NULL OR trim(request_reason) = '' THEN 1 ELSE 0 END) AS request_reason_semantic_nulls",
 '(count(*) - count(date_created)) AS date_created_nulls',
 '0 AS date_created_semantic_nulls',
 '(count(*) - count(date_modified)) AS date_modified_nulls',
 '0 AS date_modified_semantic_nulls',
 '(count(*) - count(case_close_date)) AS case_close_date_nulls',
 '0 AS case_close_date_semantic_nulls',
 '(count(*) - count(request_status)) AS request_status_nulls',
 "sum(CASE WHEN request_status IS NULL OR trim(request_status) = '' THEN 1 ELSE 0 END) 

In [54]:
', '.join(select_parts[-16:])

'(count(*) - count(`:@computed_region_ewbu_t8bu`)) AS :@computed_region_ewbu_t8bu_nulls, 0 AS :@computed_region_ewbu_t8bu_semantic_nulls, (count(*) - count(`:@computed_region_k37d_then`)) AS :@computed_region_k37d_then_nulls, 0 AS :@computed_region_k37d_then_semantic_nulls, (count(*) - count(`:@computed_region_m56f_hbma`)) AS :@computed_region_m56f_hbma_nulls, 0 AS :@computed_region_m56f_hbma_semantic_nulls, (count(*) - count(`:@computed_region_7fw3_kdpf`)) AS :@computed_region_7fw3_kdpf_nulls, 0 AS :@computed_region_7fw3_kdpf_semantic_nulls, (count(*) - count(`:@computed_region_spev_d8jm`)) AS :@computed_region_spev_d8jm_nulls, 0 AS :@computed_region_spev_d8jm_semantic_nulls, (count(*) - count(`:@computed_region_sikx_bdeb`)) AS :@computed_region_sikx_bdeb_nulls, 0 AS :@computed_region_sikx_bdeb_semantic_nulls, (count(*) - count(`:@computed_region_evki_aju8`)) AS :@computed_region_evki_aju8_nulls, 0 AS :@computed_region_evki_aju8_semantic_nulls, (count(*) - count(`:@computed_region_u4y

In [ ]:
'(count(*) - count(Location)) AS location_nulls'

In [20]:
example = nola.get("2jgv-pqrq")

In [21]:
example

[{'service_request': '2021-847416',
  'request_type': "Mayor's Request",
  'request_reason': 'Requests to the Mayor',
  'date_created': '2021-12-10T21:22:27.000',
  'date_modified': '2022-07-08T14:30:48.000',
  'case_close_date': '2022-07-08T09:30:48.000',
  'request_status': 'Closed',
  'responsible_agency': 'Executive Office of the Mayor',
  'status': 'Resolved',
  'rowid': '847416',
  'longitude': '0.0',
  'latitude': '0.0',
  'geocoded_column': {'latitude': '0.0', 'longitude': '0.0'}},
 {'service_request': '2022-858955',
  'request_type': 'Tax and Revenue',
  'request_reason': 'Occupational License Tax',
  'date_created': '2022-02-04T15:39:45.000',
  'date_modified': '2022-02-22T07:09:55.000',
  'case_close_date': '2022-02-22T01:09:55.000',
  'request_status': 'Closed',
  'responsible_agency': 'Bureau of Revenue',
  'status': 'Resolved',
  'rowid': '858955',
  'longitude': '0.0',
  'latitude': '0.0',
  'geocoded_column': {'latitude': '0.0', 'longitude': '0.0'}},
 {'service_request'

In [22]:
pd.DataFrame.from_records(example)

,service_request,request_type,request_reason,date_created,date_modified,case_close_date,request_status,responsible_agency,status,rowid,...,:@computed_region_ewbu_t8bu,:@computed_region_k37d_then,:@computed_region_m56f_hbma,:@computed_region_7fw3_kdpf,:@computed_region_spev_d8jm,:@computed_region_sikx_bdeb,:@computed_region_evki_aju8,:@computed_region_u4yh_3wk9,contractor,contractor_action
0,2021-847416,Mayor's Request,Requests to the Mayor,2021-12-10T21:22:27.000,2022-07-08T14:30:48.000,2022-07-08T09:30:48.000,Closed,Executive Office of the Mayor,Resolved,847416,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-858955,Tax and Revenue,Occupational License Tax,2022-02-04T15:39:45.000,2022-02-22T07:09:55.000,2022-02-22T01:09:55.000,Closed,Bureau of Revenue,Resolved,858955,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-1145120,Traffic Safety,Request for Traffic Calming due to Speeding,2024-10-30T12:18:29.000,2024-12-22T16:52:36.000,NaN,Pending,Department of Public Works,Pending,1145120,...,7540,1,46,39,3772,46,7,7540,NaN,NaN
3,2024-1145799,Trash/Recycling,Missed Trash Pick-Up,2024-11-02T07:58:56.000,2024-12-22T16:51:17.000,2024-11-02T04:09:40.000,Closed,Department of Sanitation,Invalid Request,1145799,...,12480,2,232,13,3455,232,4,12480,IV Waste,Invalid Request
4,2024-1145838,Roads and Streets,Push-up/Pavement Expansion,2024-11-02T15:10:10.000,2024-12-22T16:48:43.000,NaN,Pending,Department of Public Works,Pending,1145838,...,5909,1,235,50,3772,235,7,5909,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2021-783295,Trash/Recycling,Replace Trash Cart,2021-06-06T17:41:40.000,2021-10-20T05:02:17.000,2021-06-29T06:12:34.000,Closed,Department of Sanitation,Resolved,783295,...,1563,2,54,67,3457,54,3,1563,Richards Disposal,NaN
996,2021-783239,Trash/Recycling,Order Recycling Cart,2021-06-06T10:51:39.000,2021-10-20T05:02:22.000,2021-06-22T06:37:10.000,Closed,Department of Sanitation,Resolved,783239,...,5556,4,97,60,3770,97,7,5556,NaN,NaN
997,2021-783249,Trash/Recycling,Illegal Dumping,2021-06-06T12:32:33.000,2021-12-07T21:24:56.000,2021-12-07T15:24:56.000,Closed,Department of Sanitation,Duplicate Case,783249,...,757,5,181,68,4149,181,1,757,NaN,NaN
998,2021-783285,Roads/Drainage,Debris in Roadway,2021-06-06T16:32:00.000,2022-11-23T10:14:47.000,2022-11-23T04:14:47.000,Closed,Department of Public Works,Captured by Project,783285,...,13008,1,235,50,3772,235,7,13008,NaN,NaN


In [55]:
nola.get("2jgv-pqrq", select = ', '.join(select_parts[-16:]))

HTTPError: 400 Client Error: Bad Request.
	Could not parse SoQL query "select (count(*) - count(`:@computed_region_ewbu_t8bu`)) AS :@computed_region_ewbu_t8bu_nulls, 0 AS :@computed_region_ewbu_t8bu_semantic_nulls, (count(*) - count(`:@computed_region_k37d_then`)) AS :@computed_region_k37d_then_nulls, 0 AS :@computed_region_k37d_then_semantic_nulls, (count(*) - count(`:@computed_region_m56f_hbma`)) AS :@computed_region_m56f_hbma_nulls, 0 AS :@computed_region_m56f_hbma_semantic_nulls, (count(*) - count(`:@computed_region_7fw3_kdpf`)) AS :@computed_region_7fw3_kdpf_nulls, 0 AS :@computed_region_7fw3_kdpf_semantic_nulls, (count(*) - count(`:@computed_region_spev_d8jm`)) AS :@computed_region_spev_d8jm_nulls, 0 AS :@computed_region_spev_d8jm_semantic_nulls, (count(*) - count(`:@computed_region_sikx_bdeb`)) AS :@computed_region_sikx_bdeb_nulls, 0 AS :@computed_region_sikx_bdeb_semantic_nulls, (count(*) - count(`:@computed_region_evki_aju8`)) AS :@computed_region_evki_aju8_nulls, 0 AS :@compu

In [43]:
nola.get("2jgv-pqrq", select = 
        """
        (count(*) - count(service_request)) AS service_request_nulls,
        sum(CASE WHEN service_request IS NULL OR trim(service_request) = '' THEN 1 ELSE 0 END) AS service_request_semantic_nulls,
        (count(*) - count(request_type)) AS request_type_nulls,
        sum(CASE WHEN request_type IS NULL OR trim(request_type) = '' THEN 1 ELSE 0 END) AS request_type_semantic_nulls,
        (count(*) - count(request_reason)) AS request_reason_nulls,
        sum(CASE WHEN request_reason IS NULL OR trim(request_reason) = '' THEN 1 ELSE 0 END) AS request_reason_semantic_nulls,
        (count(*) - count(date_created)) AS date_created_nulls,
        0 AS date_created_semantic_nulls,
        (count(*) - count(date_modified)) AS date_modified_nulls,
        0 AS date_modified_semantic_nulls,
        (count(*) - count(case_close_date)) AS case_close_date_nulls,
        0 AS case_close_date_semantic_nulls,
        (count(*) - count(request_status)) AS request_status_nulls,
        sum(CASE WHEN request_status IS NULL OR trim(request_status) = '' THEN 1 ELSE 0 END) AS request_status_semantic_nulls,
        (count(*) - count(responsible_agency)) AS responsible_agency_nulls,
        sum(CASE WHEN responsible_agency IS NULL OR trim(responsible_agency) = '' THEN 1 ELSE 0 END) AS responsible_agency_semantic_nulls,
        (count(*) - count(final_address)) AS final_address_nulls,
        sum(CASE WHEN final_address IS NULL OR trim(final_address) = '' THEN 1 ELSE 0 END) AS final_address_semantic_nulls,
        (count(*) - count(address_councildis)) AS address_councildis_nulls,
        sum(CASE WHEN address_councildis IS NULL OR trim(address_councildis) = '' THEN 1 ELSE 0 END) AS address_councildis_semantic_nulls,
        (count(*) - count(status)) AS status_nulls,
        sum(CASE WHEN status IS NULL OR trim(status) = '' THEN 1 ELSE 0 END) AS status_semantic_nulls,
        (count(*) - count(contractor)) AS contractor_nulls,
        sum(CASE WHEN contractor IS NULL OR trim(contractor) = '' THEN 1 ELSE 0 END) AS contractor_semantic_nulls,
        (count(*) - count(contractor_action)) AS contractor_action_nulls,
        sum(CASE WHEN contractor_action IS NULL OR trim(contractor_action) = '' THEN 1 ELSE 0 END) AS contractor_action_semantic_nulls,
        (count(*) - count(rowid)) AS rowid_nulls,
        0 AS rowid_semantic_nulls,
        (count(*) - count(final_x)) AS final_x_nulls,
        0 AS final_x_semantic_nulls,
        (count(*) - count(final_y)) AS final_y_nulls,
        0 AS final_y_semantic_nulls,
        (count(*) - count(longitude)) AS longitude_nulls,
        0 AS longitude_semantic_nulls,
        (count(*) - count(latitude)) AS latitude_nulls,
        0 AS latitude_semantic_nulls
        """)

[{'service_request_nulls': '0',
  'service_request_semantic_nulls': '0',
  'request_type_nulls': '1757',
  'request_type_semantic_nulls': '1757',
  'request_reason_nulls': '1778',
  'request_reason_semantic_nulls': '1778',
  'date_created_nulls': '0',
  'date_created_semantic_nulls': '0',
  'date_modified_nulls': '0',
  'date_modified_semantic_nulls': '0',
  'case_close_date_nulls': '228903',
  'case_close_date_semantic_nulls': '0',
  'request_status_nulls': '0',
  'request_status_semantic_nulls': '0',
  'responsible_agency_nulls': '6588',
  'responsible_agency_semantic_nulls': '6588',
  'final_address_nulls': '13205',
  'final_address_semantic_nulls': '13205',
  'address_councildis_nulls': '328991',
  'address_councildis_semantic_nulls': '328991',
  'status_nulls': '463342',
  'status_semantic_nulls': '463342',
  'contractor_nulls': '722534',
  'contractor_semantic_nulls': '722534',
  'contractor_action_nulls': '850469',
  'contractor_action_semantic_nulls': '850469',
  'rowid_nulls':

In [2]:
domain = Domain("data.weho.gov")

In [3]:
ids = domain.city_datasets_ids()

INFO:data.weho.gov:Task Successful for data.weho.gov: datasets_generator


In [4]:
domain.metadata("twtae")

INFO:data.weho.gov:Task Started for data.weho.gov: metadata
ERROR:data.weho.gov:Final failure in for data.weho.gov's metadata: HTTPSConnectionPool(host='data.weho.gov', port=443): Max retries exceeded with url: /api/views/twtae.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001B87279A5D0>: Failed to resolve 'data.weho.gov' ([Errno 11001] getaddrinfo failed)"))


ConnectionError: HTTPSConnectionPool(host='data.weho.gov', port=443): Max retries exceeded with url: /api/views/twtae.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001B87279A5D0>: Failed to resolve 'data.weho.gov' ([Errno 11001] getaddrinfo failed)"))

In [12]:
df = pd.DataFrame.from_records(df)

In [13]:
df

,pib_file_number,date_occurred,originating_bureau,division_level,division,unit,working_status,shift,investigation_status,disposition,...,distance_between,subject_gender,subject_ethnicity,subject_age,subject_build,subject_height,subject_injured,subject_hospitalized,subject_arrested,subject_arrest_charges
0,FTN2021-0378,2021-12-01T00:00:00.000,FOB - Field Operations Bureau,8th District,Staff,Staff Other,Regular Working,Between 3pm-11pm,Initial,Pending,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FTN2025-0356,2025-10-28T00:00:00.000,FOB - Field Operations Bureau,6th District,Day Watch,Day Watch,Unknown Working Status,Unknown Shift Hours,Completed,UOF Justified,...,*Direct Contact,Male,Black,33,Medium,6'1'' to 6'3'',No,No,No,NaN
2,FTN2016-0524,2016-10-20T00:00:00.000,FOB - Field Operations Bureau,1st District,2nd Platoon,NaN,Unknown Working Status,Unknown Shift Hours,Completed,Use Of Force Authorized,...,0 feet to 1 feet,Male,Black,57,Small,5'4'' to 5'6'',No,No,Yes,NaN
3,FTN2016-0519,2016-10-20T00:00:00.000,FOB - Field Operations Bureau,7th District,Day Watch,Patrol,Regular Working,Between 7am-3pm,Completed,Use Of Force Authorized,...,11 feet to 14 feet,Male,Black,25,Medium,5'7'' to 5'9'',No,No,No,NaN
4,FTN2016-0523,2016-10-22T00:00:00.000,FOB - Field Operations Bureau,2nd District,A Platoon,NaN,NaN,NaN,Completed,Use Of Force Authorized,...,1 feet to 3 feet,Male,Black,27,Small,5'10'' to 6'0'',No,No,Yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,FTN2016-0053,2016-01-28T00:00:00.000,FOB - Field Operations Bureau,1st District,1st Platoon,Patrol,NaN,NaN,Completed,Use Of Force Authorized,...,1 feet to 3 feet,Male,Black,29,Medium,5'10'' to 6'0'',No,Yes,No,NaN
996,FTN2016-0054,2016-01-28T00:00:00.000,FOB - Field Operations Bureau,7th District,Day Watch,NaN,Unknown Working Status,Unknown Shift Hours,Completed,Use Of Force Authorized,...,1 feet to 3 feet | 1 feet to 3 feet,Male | Male,Black | Black,16 | 16,Small | Small,5'4'' to 5'6'' | 5'4'' to 5'6'',Yes | Yes,Yes | Yes,Yes | Yes,NaN
997,FTN2016-0056,2016-01-29T00:00:00.000,FOB - Field Operations Bureau,6th District,C Platoon,NaN,NaN,NaN,Completed,Use Of Force Authorized,...,0 feet to 1 feet,Female,Black,28,Medium,5'0'' to 5'3'',Yes,Yes,Yes,NaN
998,FTN2016-0055,2016-01-29T00:00:00.000,FOB - Field Operations Bureau,3rd District,Task Force A,Patrol,NaN,NaN,Completed,Use Of Force Authorized,...,NaN,Sex-Unk,NaN,NaN,NaN,NaN,No,No,No,NaN


In [5]:
nola = Socrata('data.nola.gov', None)

In [7]:
nola.datasets()

SSLError: HTTPSConnectionPool(host='data.nola.org', port=443): Max retries exceeded with url: /api/catalog/v1?domains=data.nola.org&offset=0 (Caused by SSLError(SSLCertVerificationError(1, "[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'data.nola.org'. (_ssl.c:1028)")))

In [6]:
nola.datasets(limit = 2)

[{'resource': {'name': 'NOPD Use of Force Incidents',
   'id': '9mnw-mbde',
   'resource_name': None,
   'parent_fxf': [],
   'description': 'This dataset represents use of force incidents by the New Orleans Police Department reported per NOPD Use of Force policy.  This dataset includes initial reports that may be subject to change through the review process.  This dataset reflects the most current status and information of these reports.  This dataset includes one row of data for each use of force incident, with information about the officers and subjects involved flattented into the incident row.  That is, the officer and subject-specific columns will contain information about all the officers and subjects, joined by the "|" character.  For example, if during a use of force incident two officers used force and three people were the subject of force, the, "Officer Age" column might contain "43 | 27", while the "Subject Age" column might contain "27 | 26 | 31".  For all officer and sub

In [7]:
nola.get_metadata("9mnw-mbde")

{'id': '9mnw-mbde',
 'name': 'NOPD Use of Force Incidents',
 'assetType': 'dataset',
 'attribution': 'Police Department (NOPD)',
 'averageRating': 0,
 'category': 'Public Safety and Preparedness',
 'createdAt': 1470193641,
 'description': 'This dataset represents use of force incidents by the New Orleans Police Department reported per NOPD Use of Force policy.  This dataset includes initial reports that may be subject to change through the review process.  This dataset reflects the most current status and information of these reports.  This dataset includes one row of data for each use of force incident, with information about the officers and subjects involved flattented into the incident row.  That is, the officer and subject-specific columns will contain information about all the officers and subjects, joined by the "|" character.  For example, if during a use of force incident two officers used force and three people were the subject of force, the, "Officer Age" column might contai